# Exploratory Data Analysis

Author: Justin Winkler

Date: September 25, 2026

The purpose of this notebook is to explore the Zillow and data center location datasets.

Effective exploration will satisfy the following criteria:

1. *Profiles* the data:
    - What size is the data?
    - What type(s) does each column contain?
    - What do the distributions of each variable look like?
2. Evaluates the *completeness* of the data:
    - Are there any missing values in the dataset? If so:
        - How many and where?
        - Why might those values be missing?
        - Based on the answer to the previous question, what is the best way to handle the missing values?
3. Ensures the *accuracy* of the data:
    - Is the data consistent with other trusted sources?
4. Verifies the *consistency* of data:
    - Are similar measurements recorded in consistent units?
    - Are observations duplicated across datasets? If so, do they match?
5. Enforces the *integrity* of the data:
    - Are IDs unique? Are they consistent between datasets?
6. Documents the data's *lineage and provenance*:
    - Where did the data come from?
    - How has the data been transformed?

Throughout the data cleaning process below, we highlight the steps taken to ensure each of the above criteria has been addressed with Python comments.

## **Import essential data processing utilities**

In [1]:
import numpy as np
import pandas as pd

## **Data loading and preprocessing**

First, define a local path to the `data` directory.

In [2]:
from pathlib import Path

data_dir = Path("data").resolve()

#### Load data center location information

Rows in `data_centers.csv` correspond to unique data center locations in the U.S. and abroad. The full dataset contains numerical fields describing the energy consumption and output of each data center, categorical fields containing tags that describe each data center's user and owner, and other data outside the scope of this project.

First, limit data centers to only U.S. locations to match the Zillow dataset. Then, select only the columns pertaining to data center names (a unique identifier for this dataset) and their addresses.

In [3]:
raw_data_centers = pd.read_csv(data_dir / "raw" / "epoch_ai" / "data_centers.csv")
us_data_centers = raw_data_centers[raw_data_centers["Country"] == "United States"]
us_data_center_locations = us_data_centers[["Name", "Address"]]

#### Load data center construction timeline information

Rows in `data_center_timelines.csv` correspond to aggregations of data center construction update news headlines. Fields in this dataset include the name of the data center the construction update belongs to, a description of the update, a count of the number of operational buildings at the data center site at the time of the update, as well as information about the energy consumption and cost of the data center.

Select only the columns pertaining to data center names (a unique identifier for this dataset), the date of the headline aggregation, and the number of buildings that were operational at the time of the aggregation. Rename the "Data center" field to "Name" and the "Buildings operational" field to "BuildingsOperational" to standardize the naming scheme for columns. Standardize date-like fields by converting them to datetime.

In [4]:
data_center_timelines = pd.read_csv(data_dir / "raw" / "epoch_ai" / "data_center_timelines.csv")
data_center_timelines = data_center_timelines[["Data center", "Date", "Buildings operational"]]

# Consistency:
data_center_timelines = data_center_timelines.rename(columns={'Data center': 'Name', "Buildings operational": "BuildingsOperational"})
data_center_timelines["Date"] = pd.to_datetime(data_center_timelines.Date)

## **Data cleaning and feature engineering**

Now that the study's raw data has been loaded, we will analyze its quality and clean the dataset accordingly before creating features useful for the study's purpose of examining the effect of data center construction on nearby home values.

This dataset poses multiple challenges. Consider a naive approach to this problem, which might involve normalizing value appreciation over one year by representing it as a percentage of average home value by zip code the month that a data center is constructed in that area. One issue with this approach is that, because many frontier AI data centers have been constructed in the last year, a sizable proportion of this dataset does not have a full year of home value appreciation data.

Additionally, when comparing the average home value of a zip code with a new data center (the *treatment group*) to the average home value of zip codes without a new data center (the *control group*), we need to be careful to choose zip codes that are comparable to the treated zip code or risk introducing a confounding variable like proximity to major cities, school quality, zoning, etc.

The solution I have settled on after consulting Claude about my concerns is to use a log change metric instead of a simple rate of change metric — i.e., calculate `ln(V_1/V_0)` where `V_1` is the average home value at some date before or after the first day of the center's operation and `V_0` is the average home value on the first day of the center's operation. While this measurement normalizes the data (so that one can compare the rate of appreciation of, say, $100,000 houses to $1,000,000 houses), the natural log also allows for symmetric arithmetic. This makes it possible to perform the following two calculations:

- `RelLogValue = ln(V_t/V_0)`: For every month `t` from 24 months before the anchor to 24 months after it, the zip's cumulative log change in home value since the anchor month. It is 0 at `t = 0` by construction, and because it is a log difference, the months before the anchor and the months after it are measured on the same footing. This is computed identically for treated zips and for their control zips.
- `Gap = RelLogValue_treated - RelLogValue_control` (computed in `data_visualization.py`): At each month, the treated zip's cumulative change net of its matched control zips' cumulative change over the same calendar months. Differencing against controls removes shocks the treated zip shares with its metro (interest rates, a regional boom), and matching controls on their pre-anchor trend handles time-invariant differences between zips (proximity to major cities, school quality, zoning). If the matching works, the gap should hover near 0 in the months *before* the anchor, which is checked visually in the event-study plot; any departure from 0 *after* the anchor is the estimated effect of the data center.

Furthermore, to ensure comparisons are fair, choose K = 5 comparable zip codes for each treated zip code according to three criteria:
- same metro area, or the same state for the few treated zips that Zillow does not assign to a metro
- similar home value at the anchor month (`Level = ln(V_0)`)
- similar trend over the 24 months leading up to the anchor month (`PreTrend = ln(V_0/V_-1)`, where `V_-1` is the value 24 months before the anchor)

The last two features are z-scored within each treated zip's pool of candidates, and the 5 candidates closest to the treated zip by Euclidean distance in that standardized space become its control group.

This study design is called a difference of differences and is a common statistical technique used to compared the causal effect of a specific treatment (in this case, the construction of data centers) on a measure (home values) between a treatment group (zip codes containing a new frontier AI data center) and a control group (comparable zip codes without new data centers).

#### Define the event of interest

Set whether to study the impact of the "first headline" about data center construction or the completion of the first data center's construction on surrounding home values.

In [5]:
from study_parameters import EventOfInterest, EVENT_OF_INTEREST

if EVENT_OF_INTEREST == EventOfInterest.FIRST_OPERATIONAL:
    # If looking for first operational date, remove dates with no operational buildings
    data_center_timelines = data_center_timelines[data_center_timelines["BuildingsOperational"] != 0]

data_center_dates_of_interest = data_center_timelines.groupby(by="Name")["Date"].min()
data_center_dates_of_interest.name = "DateOfInterest (DOI)"
print(data_center_dates_of_interest)

Name
AWS Berwick                      2021-07-01
AWS New Albany                   2022-07-01
Alibaba Zhangbei                 2024-07-23
Amazon Madison Mega Site         2023-10-02
Amazon Ridgeland                 2024-02-24
                                    ...    
Start Campus Sines Data Campus   2022-01-01
Stream Phoenix                   2022-10-30
VNET Bayin Ulanqab               2024-03-15
Vantage TX1                      2023-12-15
xAI QTS Atlanta                  2024-01-23
Name: DateOfInterest (DOI), Length: 86, dtype: datetime64[ns]


#### Combine the data center datasets

Create a unified dataset by joining the data center DataFrames on their name and keeping all columns of the trimmed datasets.

This project is interested in grouping home values by zip code because that is what is most commonly available in the Zillow dataset. Extract the zip code of data centers from their address columns.

This is one of the primary datasets, so this is an appropriate time to profile the data we have. Print the dimensions and types of the dataframe and record the number of missing values in each column.

There are a number of missing zip codes that can easily be manually filled in with information from Google Maps. We will add those manually, then drop any elements that are still missing.

For the purposes of this project, if a zip code has more than one frontier AI data center, then the effect will only be observed when the first one is constructed. For data center elements that contain duplicate zip codes, keep the one with the earliest DOI.

In [6]:
# Join the two primary datasets
data_centers = pd.merge(us_data_center_locations, data_center_dates_of_interest, on="Name", how="outer")

# Consistency/Accuracy:
# Extract the last 5-digit run of characters (preceded by whitespace) in the "Address" field to the "ZipCode" field.
data_centers["ZipCode"] = data_centers['Address'].str.extract(r'(?<=\s)(\d{5})(?!.*\d)')

# Profiling:
print("Data Center Dates of Interest (DOI)")
print("-----------------------------------")
print(f"Dimensions: {data_centers.shape}")
print()
print(f"Data Types:\n{data_centers.dtypes}")
print()
print(f"Missing Values:\n{data_centers.isnull().sum()}")
print()

# Completeness/Accuracy:
# Manually map a few data centers to their zip, sourced using Google Maps.
data_centers["ZipCode"] = data_centers["ZipCode"].fillna(data_centers["Name"].map({
    "AWS New Albany": "43054",
    "Google The Dalles": "97058",
    "Meta Huntsville": "35810",
    "CoreWeave Chester VA": "23836",
    "Microsoft-Nebius New Jersey": "08361",
    "Stream Phoenix": "85338",
    "Amazon Ridgeland": "39157",
    "Amazon Madison Mega Site": "39046",
}))

# Completeness:
# Drop any remaining rows with NaN zip codes
data_centers = data_centers.dropna(subset=['ZipCode'])

# Integrity (since we will claim that the DOI is the catalyst for change in nearby home values):
# If a ZipCode has multiple dates of interest because more than one data center belongs to it, choose the earliest
data_centers = (
    data_centers.sort_values("DateOfInterest (DOI)")
    .groupby("ZipCode", as_index=False)
    .agg({"Name": list, "Address": "first", "DateOfInterest (DOI)": "first"})
)

Data Center Dates of Interest (DOI)
-----------------------------------
Dimensions: (86, 4)

Data Types:
Name                            object
Address                         object
DateOfInterest (DOI)    datetime64[ns]
ZipCode                         object
dtype: object

Missing Values:
Name                     0
Address                 19
DateOfInterest (DOI)     0
ZipCode                 27
dtype: int64



#### Visualize the cleaned dataset

Now that a few steps have been taken to improve the quality of the data for the purposes of this project, profile and visualize the new dataset.

In [7]:
# Profiling:
print("Data Center Dates of Interest (DOI)")
print("-----------------------------------")
print(f"Dimensions: {data_centers.shape}")
print()
print(f"Data Types:\n{data_centers.dtypes}")
print()
print(f"Missing Values:\n{data_centers.isnull().sum()}")
print()

# Visualize the new data
print(data_centers.head())

Data Center Dates of Interest (DOI)
-----------------------------------
Dimensions: (57, 4)

Data Types:
ZipCode                         object
Name                            object
Address                         object
DateOfInterest (DOI)    datetime64[ns]
dtype: object

Missing Values:
ZipCode                 0
Name                    0
Address                 0
DateOfInterest (DOI)    0
dtype: int64

  ZipCode                                           Name  \
0   08361                  [Microsoft-Nebius New Jersey]   
1   14012  [Core42 Lake Mariner, Anthropic Lake Mariner]   
2   18603                                  [AWS Berwick]   
3   20109                   [STACK Infrastructure NVA02]   
4   20136                               [Google Bristow]   

                                      Address DateOfInterest (DOI)  
0    3963 S Lincoln Ave, Vineland, New Jersey           2024-05-31  
1              7725 Lake Rd, Barker, NY 14012           2024-10-01  
2        1125 Electron

#### Load Zillow home estimates

Rows in this dataset correspond to unique U.S. zip codes. The first few columns of the dataset are categorical, specifying the city, county, state, and metro area that the zip code belongs to, while the remainder of the columns are numeric and specify the Zillow Home Value Index, a "seasonally adjusted measure of the *typical* single-family home in the 35th-65th percentile," if one exists, for each month in the period from January 2000 to July 2026.

Separate the columns pertaining to zip codes and monthly average home values from zip code geographic information. Rename the "RegionName" field to "ZipCode". Standardize date-like column names by converting them to datetime.

Print the first five entries of the dataset for a preview of the data's structure.

In [8]:
# Read then concatenate the two zestimate datasets. There are two datasets to accommodate
# GitHub's file size limit.
zestimates1 = pd.read_csv(data_dir / "raw" / "zillow" / "zestimates_by_zip_1.csv")
zestimates2 = pd.read_csv(data_dir / "raw" / "zillow" / "zestimates_by_zip_2.csv")
zestimates = pd.concat([zestimates1, zestimates2], ignore_index=True)

# Split the data into two DataFrame objects:
#   - `zip_geo`: categorical features describing geographical information about zip codes
#   - `zestimates_by_zip`: a chronological breakdown of average home value zestimates by zip code
zestimates_by_zip = zestimates.filter(regex=r'RegionName|\d{4}-\d{2}-\d{2}')
zip_geo = zestimates.filter(regex=r'RegionName|State(?!Name)|Metro|CountyName|SizeRank')

# Consistency:
# Rename "RegionName" columns to "ZipCode" and standardize zip codes to 5-character strings.
# Pad the front with 0s, if necessary, since they are originally stored as int64s.
zestimates_by_zip = zestimates_by_zip.rename(columns={"RegionName": "ZipCode"})
zip_geo = zip_geo.rename(columns={"RegionName": "ZipCode"})
zip_geo["ZipCode"] = zip_geo["ZipCode"].astype(str).str.zfill(5)

# Consistency:
# Standardize date-like column names
zestimates_by_zip.columns = ["ZipCode", *pd.to_datetime(zestimates_by_zip.columns[1:])]

# Visualize the new data
print(zestimates_by_zip.head())

   ZipCode  2000-01-31 00:00:00  2000-02-29 00:00:00  2000-03-31 00:00:00  \
0    77494        212188.981819        212373.142689        212865.242558   
1     8701        113571.789919        114040.532783        114357.352733   
2    77449        105369.930306        105384.490172        105255.149979   
3    11368        171194.526575        172896.026224        174174.055641   
4    77084        105256.097011        105211.953674        105025.045819   

   2000-04-30 00:00:00  2000-05-31 00:00:00  2000-06-30 00:00:00  \
0        213859.178025        213894.730929        213739.535344   
1        115159.988256        115995.021610        116974.287103   
2        105245.255125        105293.299359        105488.782188   
3        176307.350060        177837.295188        179501.272127   
4        104939.654419        104914.252908        105054.918073   

   2000-07-31 00:00:00  2000-08-31 00:00:00  2000-09-30 00:00:00  ...  \
0        212973.219993        213006.239062        2127

#### Create the treated panel

The next step in the analysis is to create a panel, or chronological record of a certain metric over a period of time, of the proposed transformation of home value estimates, `ln(V_1/V_0)`. To do this, we will need to combine the information in the historical Zillow value estimate dataset and the data center construction dataset.

First, create a new dataframe object, `zestimates_by_month`, which contains rows representing snapshots of home value estimates for individual months.

Define the "anchor" for each zip code in the treatment group. These will serve as `t_0` for data center construction events and are defined as the date on which the last Zillow estimate was released for a treated zip code before the data center's event of interest (the first headline about construction or the first day of a center's operation, depending on the event of interest set above).

Finally, using the anchor, create the final `treated_panel` which tracks the value of `ln(V_1/V_0)` for each treated zip code from 24 months *before* the zip code's anchor date to 24 months *after* the anchor date or until the last recorded Zillow estimate if the anchor date for a given zip code is from the last 24 months.

In [9]:
# DISCLAIMER: The following section was created with help from Claude Code's Opus 5 model.

# -------------------------- CREATED WITH HELP FROM CLAUDE CODE --------------------------

from study_parameters import EVENT_WINDOW_MONTHS

# Reshape the wide Zillow table into one row per zip code per month.
zestimates_by_month = zestimates_by_zip.melt(id_vars="ZipCode", var_name="Date", value_name="Zestimate")
zestimates_by_month = zestimates_by_month.dropna(subset="Zestimate")

# Consistency/Accuracy:
# Zillow stores zip codes as integers, so zero-pad them to match the five-character strings 
# in data_centers.
zestimates_by_month["ZipCode"] = zestimates_by_month["ZipCode"].astype(str).str.zfill(5)

# Consistency:
# Standardize the Date column as datetime
zestimates_by_month["Date"] = pd.to_datetime(zestimates_by_month["Date"])

# Lineage and provenance:
# Transform Zillow home value estimates to make arithmetic symmetric
zestimates_by_month["LogValue"] = np.log(zestimates_by_month["Zestimate"])

# Sort entries chronologically to match "panel" model and for viewing purposes
zestimates_by_month = zestimates_by_month.sort_values("Date")

# Anchor each treated zip code to the last Zillow observation strictly before its date of
# interest. merge_asof requires both frames to be sorted on the matched date.
anchors = pd.merge_asof(
    data_centers[["ZipCode", "DateOfInterest (DOI)"]].sort_values("DateOfInterest (DOI)"),
    zestimates_by_month[["ZipCode", "Date"]],
    left_on="DateOfInterest (DOI)",
    right_on="Date",
    by="ZipCode",
    direction="backward",
    allow_exact_matches=False,
)
anchors = anchors.rename(columns={"Date": "AnchorDate"}).dropna(subset="AnchorDate")

# Restrict the panel to treated zip codes
treated_panel = zestimates_by_month.merge(anchors[["ZipCode", "AnchorDate"]], on="ZipCode")

# Express each observation's date in months relative to its anchor (EventTime = 0) 
treated_panel["EventTime"] = (
    12 * (treated_panel["Date"].dt.year - treated_panel["AnchorDate"].dt.year)
    + (treated_panel["Date"].dt.month - treated_panel["AnchorDate"].dt.month)
)
treated_panel = treated_panel[treated_panel["EventTime"].abs() <= EVENT_WINDOW_MONTHS].copy()
baseline = treated_panel["LogValue"].where(treated_panel["EventTime"] == 0)

# Lineage and provenance
# Re-base log values to the anchor month
treated_panel["RelLogValue"] = treated_panel["LogValue"] - baseline.groupby(treated_panel["ZipCode"]).transform("first")

# -------------------------- CREATED WITH HELP FROM CLAUDE CODE --------------------------

#### Compute values for the control group selection criteria

As mentioned above, to ensure comparisons between treatment group zip codes and control group zip codes are fair, we will choose comparable control zip codes according to the following criteria:
- same metro
- similar pre-data center construction value
- similar pre-data center construction value trend

To determine which zip codes could serve as candidates for a treated zip code's control group, we must calculate values for each criterion above on a per-control-candidate basis, most notably the candidate zip code's average home value estimate and trend prior to the construction of a data center in its associated treated zip code.

For each treated zip code and for each of its candidate control zip codes (those in the same metro or state), compute the "Level" and "PreTrend" for each to represent the average home value and change in average home value over the length of the study's prescribed horizon preceding construction of the treated zip's data center, respectively. These respectively correspond to `ln(V_0)` and `ln(V_-1/V_0)` in previous cells.

Later, we will choose the "closest" matches to each treated zip code from the control candidate pools using a Euclidean distance calculation using these new features. Since these features live on vastly different scales, we should standardize the values we calculate with z-score calculations.

In [10]:
# DISCLAIMER: The following section was created with help from Claude Code's Opus 5 model.

# -------------------------- CREATED WITH HELP FROM CLAUDE CODE --------------------------

# Split zip code geographical information based on whether the zip code belongs to the 
# treatment group.
treated_zips = treated_panel["ZipCode"].unique().tolist()
treated_zip_geo = zip_geo[zip_geo["ZipCode"].isin(treated_zips)]
untreated_zip_geo = zip_geo[~zip_geo["ZipCode"].isin(treated_zips)]

# Completeness:
# Control candidate zips come from the same metro. If a treated zip code does not have a 
# metro area specified in the dataset, use the zip code's state as selection criteria 
# instead.
has_metro = treated_zip_geo["Metro"].notna()
control_candidate_pools = pd.concat([
    treated_zip_geo[has_metro].merge(untreated_zip_geo, on="Metro", suffixes=("", "ControlCandidate")),
    treated_zip_geo[~has_metro].merge(untreated_zip_geo, on="State", suffixes=("", "ControlCandidate")),
])[["ZipCode", "ZipCodeControlCandidate"]].rename(columns={"ZipCode": "TreatedZip", "ZipCodeControlCandidate": "ControlCandidateZip"})

def features_at_anchor(pairs, zip_col):
    # Log price level at the anchor month, and its change over the preceding window.
    pairs = pairs.copy()
    pairs["PreDate"] = pairs["AnchorDate"] - pd.DateOffset(months=EVENT_WINDOW_MONTHS) + pd.offsets.MonthEnd(0)
    values = zestimates_by_month[["ZipCode", "Date", "LogValue"]]
    pairs = pairs.merge(values.rename(columns={"ZipCode": zip_col, "Date": "AnchorDate", "LogValue": "Level"}), on=[zip_col, "AnchorDate"], how="left")
    pairs = pairs.merge(values.rename(columns={"ZipCode": zip_col, "Date": "PreDate", "LogValue": "PreLevel"}), on=[zip_col, "PreDate"], how="left")
    pairs["PreTrend"] = pairs["Level"] - pairs["PreLevel"]
    return pairs.drop(columns=["PreDate", "PreLevel"])

# Feature engineering:
# For each treated zip code and for each of its candidate control zip codes (those in the same metro or state),
# compute the "Level" and "PreTrend" for each to represent the average home value and change in average home 
# value over the length of the study's prescribed horizon preceding construction of the treated zip's data 
# center.
treated_anchors = anchors.rename(columns={"ZipCode": "TreatedZip"})[["TreatedZip", "AnchorDate"]]
control_candidate_features = features_at_anchor(control_candidate_pools.merge(treated_anchors, on="TreatedZip"), "ControlCandidateZip")
control_candidate_features = control_candidate_features.dropna(subset=["Level", "PreTrend"])
treated_features = features_at_anchor(treated_anchors, "TreatedZip")

# Lineage and provenance:
# Compute the z-score for each feature within each treated zip's control candidate pool
FEATURES = ["Level", "PreTrend"]
pool_stats = control_candidate_features.groupby("TreatedZip")[FEATURES].agg(["mean", "std"])
pool_stats.columns = [f"{feature}{stat.title()}" for feature, stat in pool_stats.columns]  # LevelMean, LevelStd, ...

def z_score(frame):
    # Standardize each feature against the treated zip's own control candidate pool.
    frame = frame.merge(pool_stats, left_on="TreatedZip", right_index=True)
    for feature in FEATURES:
        frame[f"{feature}Z"] = (frame[feature] - frame[f"{feature}Mean"]) / frame[f"{feature}Std"]
    return frame.drop(columns=pool_stats.columns)

control_candidate_features = z_score(control_candidate_features)
treated_features = z_score(treated_features)

# Accuracy:
# Preview the z-scores we calculated to ensure they look reasonable.
print(control_candidate_features)
print(treated_features)

# -------------------------- CREATED WITH HELP FROM CLAUDE CODE --------------------------

     TreatedZip ControlCandidateZip AnchorDate      Level  PreTrend    LevelZ  \
0         78245               78130 2018-11-30  12.400937  0.082992  0.291096   
1         78245               78254 2018-11-30  12.336214  0.080318  0.126982   
2         78245               78249 2018-11-30  12.292721  0.103230  0.016699   
3         78245               78253 2018-11-30  12.419756  0.059701  0.338815   
4         78245               78250 2018-11-30  12.064674  0.121885 -0.561543   
...         ...                 ...        ...        ...       ...       ...   
6459      58436               58048 2023-09-30  12.268955 -0.134248 -0.164734   
6460      58436               58656 2023-09-30  12.334124  0.085810  0.057977   
6461      58436               58346 2023-09-30  11.988711 -0.034009 -1.122452   
6464      58436               58429 2023-09-30  11.957930  0.078194 -1.227643   
6472      58436               58655 2023-09-30  12.658532  0.069879  1.166626   

      PreTrendZ  
0     -0.

#### Choose control group members

With the criteria for selecting control zip codes defined and evaluated, we can choose these zip codes according to their "closeness" to their associated treated zip code based on a Euclidean distance calculation using the "Level" and "PreTrend" features.

Select only the K closest neighbors where K is a study-level parameter.

Preview the quality of the best and worst control/treatment group matches by printing the Zestimate and pre-data center value trend for each.

In [11]:
# DISCLAIMER: The following section was created with help from Claude Code's Opus 5 model.

# -------------------------- CREATED WITH HELP FROM CLAUDE CODE --------------------------

# Set the number of control zip codes to use per treated zip code
K_NEAREST = 5

# Euclidean distance from each control candidate to its treated zip in standardized feature space
z_columns = [f"{feature}Z" for feature in FEATURES]
control_candidates = control_candidate_features.merge(
    treated_features[["TreatedZip", *z_columns]], on="TreatedZip", suffixes=("", "Treated")
)
control_candidates["Distance"] = np.sqrt(sum(
    (control_candidates[column] - control_candidates[f"{column}Treated"]) ** 2 for column in z_columns
))

# Keep the K_NEAREST closest control candidates per treated zip
control_candidates = control_candidates.sort_values(["TreatedZip", "Distance"])
control_candidates["Rank"] = control_candidates.groupby("TreatedZip").cumcount() + 1
matched_controls = (
    control_candidates[control_candidates["Rank"] <= K_NEAREST][["TreatedZip", "ControlCandidateZip", "Rank", "Distance"]]
    .reset_index(drop=True)
)

# Every treated zip should have exactly K_NEAREST control candidates.
control_candidates_per_zip = matched_controls.groupby("TreatedZip").size()
assert (control_candidates_per_zip == K_NEAREST).all(), control_candidates_per_zip[control_candidates_per_zip != K_NEAREST]

# Put each treated zip's raw features beside its control candidates' so the match quality can be eyeballed.
match_review = pd.concat([
    treated_features.assign(ControlCandidateZip="(treated)", Rank=0, Distance=0.0),
    matched_controls.merge(control_candidate_features, on=["TreatedZip", "ControlCandidateZip"]),
])[["TreatedZip", "ControlCandidateZip", "Rank", "Distance", "Level", "PreTrend"]].sort_values(["TreatedZip", "Rank"])
match_review["Zestimate"] = np.exp(match_review["Level"]).round(-3)
match_review["PreTrendPct"] = (100 * (np.exp(match_review["PreTrend"]) - 1)).round(1)

# Glance at the best- and worst-matched treated zips beside their control candidates.
worst_distance = matched_controls.groupby("TreatedZip")["Distance"].max().sort_values()
for treated_zip in [worst_distance.index[0], worst_distance.index[-1]]:
    print(f"\nTreatedZip {treated_zip}")
    print(match_review[match_review["TreatedZip"] == treated_zip].drop(columns="TreatedZip").to_string(index=False))

# -------------------------- CREATED WITH HELP FROM CLAUDE CODE --------------------------


TreatedZip 68138
ControlCandidateZip  Rank  Distance     Level  PreTrend  Zestimate  PreTrendPct
          (treated)     0  0.000000 12.172923  0.125146   193000.0         13.3
              68409     1  0.017839 12.167420  0.125022   192000.0         13.3
              68127     2  0.025111 12.179752  0.124567   195000.0         13.3
              68137     3  0.036074 12.181783  0.124098   195000.0         13.2
              68157     4  0.093330 12.181374  0.120936   195000.0         12.9
              51546     5  0.100957 12.164196  0.120574   192000.0         12.8

TreatedZip 30721
ControlCandidateZip  Rank  Distance     Level  PreTrend  Zestimate  PreTrendPct
          (treated)     0  0.000000 12.305979  0.126000   221000.0         13.4
              30705     1  1.581720 12.391245  0.091423   241000.0          9.6
              30720     2  1.705040 12.444107  0.090976   254000.0          9.5
              30710     3  2.330335 12.567287  0.083666   287000.0          8.7
    

#### Construct the control panel

Now that we know *which* zip codes to compare to each zip in the treated panel, we need to compute a panel for each of the zip codes in a treated zip's control group. We do this by anchoring the control group on the treated zip's date of interest. We then re-base the control group's monthly value estimates off of the new anchor month to create an analogously timed panel.

With the panel now computed correctly for the control group, average the K panel rows into one. This will not correspond to any one zip anymore, but it provides a simple and convenient comparison for the single treated zip.

In [12]:
# DISCLAIMER: The following section was created with help from Claude Code's Opus 5 model.

# -------------------------- CREATED WITH HELP FROM CLAUDE CODE --------------------------

# Build each matched control candidate's path in its treated zip's event time. A control 
# candidate's clock is set by the treated zip's anchor, and its log values are re-based to 
# its own value at that month, so every (TreatedZip, ControlCandidateZip) pair passes 
# through 0 at EventTime 0.
control_candidate_values = zestimates_by_month[zestimates_by_month["ZipCode"].isin(matched_controls["ControlCandidateZip"])]
control_paths = (
    matched_controls.merge(treated_anchors, on="TreatedZip")
    .merge(control_candidate_values.rename(columns={"ZipCode": "ControlCandidateZip"}), on="ControlCandidateZip")
)
control_paths["EventTime"] = (
    12 * (control_paths["Date"].dt.year - control_paths["AnchorDate"].dt.year)
    + (control_paths["Date"].dt.month - control_paths["AnchorDate"].dt.month)
)
control_paths = control_paths[control_paths["EventTime"].abs() <= EVENT_WINDOW_MONTHS].copy()
baseline = control_paths["LogValue"].where(control_paths["EventTime"] == 0)
control_paths["RelLogValue"] = control_paths["LogValue"] - baseline.groupby(
    [control_paths["TreatedZip"], control_paths["ControlCandidateZip"]]
).transform("first")

# Average the K_NEAREST control candidate paths into one synthetic control path per treated zip.
control_panel = (
    control_paths.groupby(["TreatedZip", "EventTime"], as_index=False)
    .agg(RelLogValue=("RelLogValue", "mean"), ControlCount=("ControlCandidateZip", "size"))
)

# Accuracy:
# Every treated zip's control path should pass through 0 at EventTime 0, and every point
# on it should be built from all K_NEAREST control candidates.
at_anchor = control_panel[control_panel["EventTime"] == 0]
assert len(at_anchor) == treated_anchors["TreatedZip"].nunique(), "a treated zip has no control path"
assert np.allclose(at_anchor["RelLogValue"], 0), "control path is not re-based to EventTime 0"
short_handed = control_panel[control_panel["ControlCount"] < K_NEAREST]
print(f"{control_panel['TreatedZip'].nunique()} control paths, {len(control_panel)} (TreatedZip, EventTime) points")
print(f"{len(short_handed)} points built from fewer than {K_NEAREST} control candidates")
print(control_panel.groupby("EventTime")["TreatedZip"].size().rename("TreatedZipsObserved").iloc[::6].to_string())

# -------------------------- CREATED WITH HELP FROM CLAUDE CODE --------------------------

55 control paths, 2646 (TreatedZip, EventTime) points
0 points built from fewer than 5 control candidates
EventTime
-24    55
-18    55
-12    55
-6     55
 0     55
 6     55
 12    54
 18    52
 24    46


## **Save EDA results**

Save the results of EDA for later visualization.

In [13]:
# Save cleaned data to CSV
clean_data_prefix = "operational_" if EVENT_OF_INTEREST == EventOfInterest.FIRST_OPERATIONAL else "headline_"
treated_panel.to_csv(data_dir / "final"  / (clean_data_prefix + "treated_zipcode_value_estimates.csv"), index=False)
control_panel.to_csv(data_dir / "final"  / (clean_data_prefix + "control_zipcode_value_estimates.csv"), index=False)